# Módulo 02 · Aula 1 — NumPy

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

---

O módulo 01 deu a linguagem. Agora começa a ciência de dados de verdade, e ela começa
pelo **NumPy** — a biblioteca que está por baixo de praticamente tudo em Python
científico, inclusive do pandas.

Ao final desta aula você vai saber:

- o que é um **array** e por que ele existe, já que listas existem;
- fazer contas **vetorizadas**, sem escrever loops;
- indexar, fatiar e filtrar arrays com **máscaras booleanas**;
- usar funções de agregação (`mean`, `std`, `sum`, `cumprod`) e o parâmetro `axis`;
- lidar com `NaN`.

**Tempo estimado:** 60 minutos.

In [ ]:
import numpy as np

print("NumPy:", np.__version__)

## 1. O array

Um `ndarray` é uma sequência de números **do mesmo tipo**, guardada de forma contígua na
memória. Essa restrição — todos do mesmo tipo — é o que permite ao NumPy ser rápido.

In [ ]:
precos_lista = [32.50, 61.20, 28.75, 41.90, 15.30]
precos = np.array(precos_lista)

print(precos)
print(type(precos))
print(precos.dtype)      # tipo dos elementos
print(precos.shape)      # formato: 5 elementos em 1 dimensão
print(precos.size)       # total de elementos

### Por que não usar listas?

Duas razões, e a segunda é a que muda seu dia a dia.

**1. Velocidade.** O NumPy executa as operações em código compilado, sobre memória
contígua. A diferença aparece já em algumas centenas de milhares de elementos.

In [ ]:
import time

n = 1_000_000
lista = list(range(n))
array = np.arange(n)

inicio = time.time()
resultado_lista = [x * 2 for x in lista]
tempo_lista = time.time() - inicio

inicio = time.time()
resultado_array = array * 2
tempo_array = time.time() - inicio

print(f"Lista : {tempo_lista*1000:7.1f} ms")
print(f"Array : {tempo_array*1000:7.1f} ms")
print(f"NumPy foi ~{tempo_lista/max(tempo_array, 1e-9):.0f}× mais rápido")

**2. Sintaxe.** Operações aritméticas com arrays valem para todos os elementos de uma
vez. Isso se chama **vetorização**, e é a ideia central da aula.

In [ ]:
precos_lista = [32.50, 61.20, 28.75]
precos_array = np.array(precos_lista)

# com listas, é preciso um loop:
print([p * 1.1 for p in precos_lista])

# com arrays, a conta se escreve como na matemática:
print(precos_array * 1.1)

In [ ]:
# Com listas, o operador * significa outra coisa completamente:
print([1, 2, 3] * 3)        # repete a lista
print(np.array([1, 2, 3]) * 3)   # multiplica cada elemento

## 2. Criando arrays

In [ ]:
print(np.array([1, 2, 3]))                  # a partir de uma lista
print(np.zeros(5))                          # cinco zeros
print(np.ones(3))
print(np.full(4, 7.5))                      # preenchido com um valor
print(np.arange(0, 10, 2))                  # como range(), mas devolve array
print(np.linspace(0, 1, 5))                 # 5 números igualmente espaçados de 0 a 1

In [ ]:
# Arrays com números aleatórios — sempre com semente, para o resultado ser reproduzível
gerador = np.random.default_rng(42)

print(gerador.normal(loc=0, scale=0.02, size=5))    # normal: média 0, desvio 2%
print(gerador.integers(1, 7, size=5))               # inteiros de 1 a 6

> **Sempre fixe a semente** (`default_rng(42)`) quando usar aleatoriedade em análise.
> Sem isso, seu notebook produz números diferentes a cada execução e ninguém — inclusive
> você — consegue reproduzir o resultado que você relatou.

### Arrays de duas dimensões

Um array 2D é uma matriz: linhas e colunas. É a estrutura por baixo de uma tabela.

In [ ]:
# Preços de fechamento: 4 dias (linhas) × 3 ativos (colunas)
matriz = np.array([
    [32.50, 61.20, 28.75],
    [32.80, 60.10, 28.90],
    [31.90, 62.40, 29.15],
    [33.10, 61.85, 28.60],
])

print(matriz)
print("shape :", matriz.shape)    # (linhas, colunas)
print("ndim  :", matriz.ndim)     # número de dimensões
print("size  :", matriz.size)     # total de elementos

## 3. Indexação e fatiamento

As mesmas regras das listas — inclui o início, exclui o fim — mais a possibilidade de
indexar por dimensão.

In [ ]:
precos = np.array([32.50, 61.20, 28.75, 41.90, 15.30])

print(precos[0])
print(precos[-1])
print(precos[1:4])
print(precos[:3])
print(precos[::-1])

In [ ]:
# Em 2D: [linha, coluna]
print(matriz[0, 0])      # primeira linha, primeira coluna
print(matriz[2, 1])      # terceira linha, segunda coluna

print(matriz[0])         # a primeira linha inteira
print(matriz[:, 0])      # a primeira COLUNA inteira (o ":" significa "todas as linhas")
print(matriz[1:3, :2])   # linhas 1 e 2, colunas 0 e 1

> `matriz[:, 0]` é a construção mais importante desta seção: *"todas as linhas, coluna
> 0"*. Guardar a lógica `[linhas, colunas]` agora vai economizar muita confusão quando
> chegarmos em `.loc` e `.iloc` no pandas.

## 4. Máscaras booleanas: o filtro do NumPy

Comparar um array com um valor devolve um array de `True`/`False` — a **máscara**. Usar
essa máscara como índice devolve apenas os elementos onde ela é `True`.

Esse mecanismo é a base de toda filtragem de dados que faremos daqui em diante.

In [ ]:
precos = np.array([32.50, 61.20, 28.75, 41.90, 15.30])

mascara = precos > 30
print(mascara)               # array de booleanos, mesmo tamanho do original
print(precos[mascara])       # só os elementos onde a máscara é True
print(precos[precos > 30])   # o jeito compacto, tudo em uma linha

In [ ]:
# Combinando condições: use & (e), | (ou), ~ (não)
# e SEMPRE com parênteses em volta de cada condição
print(precos[(precos > 20) & (precos < 50)])
print(precos[(precos < 20) | (precos > 60)])
print(precos[~(precos > 30)])

> **Atenção:** Em NumPy (e em pandas) use `&`, `|`, `~` — **não** `and`, `or`, `not`. As
> palavras em português esperam um único `True`/`False`, e aqui temos um array inteiro
> deles. Usar `and` gera um `ValueError` com a mensagem *"truth value of an array is
> ambiguous"* — quando você vir esse erro, é isso que ele quer dizer.

In [ ]:
# Máscaras respondem perguntas quantitativas rapidamente:
print("Quantos acima de 30?", (precos > 30).sum())      # True conta como 1
print("Proporção acima de 30:", (precos > 30).mean())   # média de 0s e 1s = proporção
print("Algum acima de 60?", (precos > 60).any())
print("Todos acima de 10?", (precos > 10).all())

In [ ]:
# np.where: um "if" vetorizado — where(condicao, valor_se_verdadeiro, valor_se_falso)
retornos = np.array([0.021, -0.013, 0.004, -0.030, 0.017])

classificacao = np.where(retornos > 0, "alta", "queda")
print(classificacao)

# também serve para transformar valores condicionalmente:
print(np.where(retornos < 0, 0, retornos))   # zera os negativos

## 5. Operações e funções matemáticas

Todas são elemento a elemento, sem loop.

In [ ]:
precos = np.array([32.50, 61.20, 28.75, 41.90])
quantidades = np.array([100, 50, 200, 80])

print(precos + 1)              # escalar aplicado a todos (isto é "broadcasting")
print(precos * quantidades)    # elemento a elemento entre dois arrays
print(np.round(precos / 5.45, 2))
print(np.sqrt(precos))
print(np.log(precos))

### Agregações

| Função | O que faz |
|---|---|
| `.sum()` | soma |
| `.mean()` | média |
| `.std()` | desvio padrão |
| `.min()` / `.max()` | menor / maior |
| `.argmin()` / `.argmax()` | **posição** do menor / maior |
| `.cumsum()` / `.cumprod()` | soma / produto acumulado |

In [ ]:
print("Soma   :", precos.sum())
print("Média  :", round(precos.mean(), 2))
print("Desvio :", round(precos.std(), 2))
print("Máximo :", precos.max())
print("Posição do máximo:", precos.argmax())
print("Acumulado:", precos.cumsum())

### `axis`: agregando linhas ou colunas

Em arrays 2D é preciso dizer **em que direção** agregar:

- `axis=0` → percorre as **linhas**, produz um resultado por **coluna**;
- `axis=1` → percorre as **colunas**, produz um resultado por **linha**.

A confusão é clássica. O truque para lembrar: `axis` indica o eixo que **desaparece**.

In [ ]:
matriz = np.array([
    [32.50, 61.20, 28.75],
    [32.80, 60.10, 28.90],
    [31.90, 62.40, 29.15],
    [33.10, 61.85, 28.60],
])

print("shape original:", matriz.shape)
print()
print("Média por ativo (axis=0):", np.round(matriz.mean(axis=0), 2), " shape:", matriz.mean(axis=0).shape)
print("Média por dia   (axis=1):", np.round(matriz.mean(axis=1), 2), " shape:", matriz.mean(axis=1).shape)
print("Média geral (sem axis)  :", round(matriz.mean(), 2))

## 6. `NaN`: o buraco nos dados

`np.nan` (*Not a Number*) representa um valor ausente. Ele é contagioso: qualquer conta
que o envolva devolve `NaN`.

In [ ]:
precos = np.array([32.50, np.nan, 28.75, 41.90])

print(precos)
print("Média comum :", precos.mean())        # NaN contamina tudo
print("Média nan-safe:", np.nanmean(precos)) # ignora os ausentes
print("Onde há NaN :", np.isnan(precos))
print("Quantos NaN :", np.isnan(precos).sum())

> **Atenção:** Repare que `np.nanmean` calculou a média dos **três** valores existentes.
> Ignorar ausentes é uma decisão analítica, não um detalhe técnico: se os dados faltam por
> um motivo sistemático (só os clientes ricos informaram patrimônio, por exemplo), a média
> dos presentes é enviesada. Voltaremos a isso na aula de limpeza.

E uma curiosidade que já derrubou muito código: `NaN` não é igual a nada, nem a ele
mesmo. Por isso se testa com `np.isnan()`, nunca com `== np.nan`.

In [ ]:
print(np.nan == np.nan)
print(np.isnan(np.nan))

## 7. Aplicação: retornos e volatilidade

Vamos usar tudo isso em um problema real de finanças. Estes são preços de fechamento
fictícios de 10 pregões.

In [ ]:
precos = np.array([32.50, 32.80, 31.90, 33.10, 33.45,
                   32.95, 34.20, 34.05, 33.60, 35.10])

# Retorno diário simples: (preço_hoje - preço_ontem) / preço_ontem
# precos[1:]  -> do 2º dia em diante
# precos[:-1] -> do 1º ao penúltimo
retornos = (precos[1:] - precos[:-1]) / precos[:-1]

print("Retornos diários:")
print(np.round(retornos * 100, 2), "%")

Repare no que acabou de acontecer: **o cálculo dos retornos de uma série inteira, sem um
único loop.** `precos[1:]` e `precos[:-1]` são a mesma série deslocada em um dia, e a
subtração entre elas alinha cada dia com o anterior. Esse é o jeito de pensar do NumPy.

In [ ]:
retorno_medio = retornos.mean()
volatilidade = retornos.std(ddof=1)      # ddof=1 -> desvio padrão amostral

print(f"Retorno médio diário : {retorno_medio:.4%}")
print(f"Volatilidade diária  : {volatilidade:.4%}")

# Anualizando (252 pregões por ano)
print(f"Retorno anualizado   : {retorno_medio * 252:.2%}")
print(f"Volatilidade anual   : {volatilidade * np.sqrt(252):.2%}")

In [ ]:
# Retorno acumulado do período, via produto acumulado
acumulado = np.cumprod(1 + retornos) - 1

print("Evolução acumulada:")
for dia, valor in enumerate(acumulado, start=2):
    print(f"  dia {dia:>2}: {valor:>7.2%}")

print(f"\nRetorno total do período: {acumulado[-1]:.2%}")
print(f"Conferindo pelo preço   : {(precos[-1] - precos[0]) / precos[0]:.2%}")

In [ ]:
# Estatísticas com máscaras booleanas
dias_alta = (retornos > 0).sum()
dias_baixa = (retornos < 0).sum()

print(f"Pregões de alta  : {dias_alta} ({dias_alta/len(retornos):.0%})")
print(f"Pregões de baixa : {dias_baixa} ({dias_baixa/len(retornos):.0%})")
print(f"Maior alta       : {retornos.max():.2%} (no dia {retornos.argmax()+2})")
print(f"Maior queda      : {retornos.min():.2%} (no dia {retornos.argmin()+2})")
print(f"Retorno médio nos dias de alta: {retornos[retornos > 0].mean():.2%}")

## 8. Recapitulando

- O **array** guarda elementos do mesmo tipo e permite operações **vetorizadas** — sem
  loops, e muito mais rápido.
- Criação: `np.array`, `np.zeros`, `np.arange`, `np.linspace`,
  `np.random.default_rng(semente)`.
- Indexação 2D é `[linha, coluna]`; `:` significa "todos".
- **Máscaras booleanas** (`precos[precos > 30]`) são a base de toda filtragem. Use
  `&`, `|`, `~` com parênteses — nunca `and`/`or`/`not`.
- `np.where(condicao, se_sim, se_nao)` é o `if` vetorizado.
- `axis=0` agrega ao longo das linhas (resultado por coluna); `axis=1`, o contrário.
- `NaN` contamina qualquer conta; use `np.nanmean` e `np.isnan`.

**Próxima aula:** pandas — a mesma lógica, agora com tabelas de verdade, com nomes de
colunas e tipos diferentes por coluna.